# Layout-Aware OCR: PaddleOCR (Hugging Face) & EasyOCR for Pharmaceutical Documents

**Project 4: Advanced OCR Comparison**

In this notebook, you'll explore two powerful layout-aware OCR engines[cite: 2]. We will run **PaddleOCR** natively on Apple Silicon using the Hugging Face `transformers` pipeline, and compare it with **EasyOCR**[cite: 2]. 

## What You'll Learn
- How to set up and run PaddleOCR as a Vision-Language Model via Transformers
- How to use EasyOCR for quick, reliable text extraction[cite: 2]
- How to compare different OCR engines side-by-side[cite: 2]

**Sample Document:** [Pharmaceutical SDF - Page 4 (Certificate of Processing)](https://drive.google.com/file/d/1UvYmChia7K90-xjCsHsGqhixmDrPfhHw/view?usp=sharing)[cite: 2]

---

## Part 1: PaddleOCR (via Hugging Face)

PaddleOCR is a powerful, layout-aware OCR engine developed by Baidu[cite: 2]. By using the Hugging Face pipeline, we bypass the heavy PaddlePaddle backend and use your Mac's native hardware to evaluate the image as a Vision-Language Model (VLM).

Let's set it up!

In [1]:
# ============================================
# STEP 1: Install Dependencies
# ============================================
#
# WHAT WE'RE DOING:
# Installing MLX, Transformers, and PDF processing tools.
# ============================================

%pip install mlx transformers pillow torch torchvision
%pip install pdf2image

print("\n" + "=" * 50)
print("All dependencies installed!")
print("=" * 50)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

All dependencies installed!


In [8]:
# ============================================
# STEP 2: Import Libraries
# ============================================

from transformers import pipeline
from pdf2image import convert_from_path
from PIL import Image
import matplotlib.pyplot as plt
import os
import torch

print("All libraries imported successfully!")

All libraries imported successfully!


In [9]:
# ============================================
# STEP 3: Upload Your Pharmaceutical SDF
# ============================================
#
# WHAT WE'RE DOING:
# Uploading and converting the PDF to an image[cite: 2].
# ============================================


image_path = 'test.png'


In [11]:
# ============================================
# STEP 4: Initialize the Hugging Face Pipeline
# ============================================

print("Initializing PaddleOCR-VL via Transformers (this may download model files)...")

# Initialize the pure Transformers pipeline
pipe = pipeline("image-text-to-text", model="PaddlePaddle/PaddleOCR-VL", dtype="bfloat16")

print("\nPipeline initialized and ready!")

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


Initializing PaddleOCR-VL via Transformers (this may download model files)...


Loading weights:   0%|          | 0/608 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


The repository PaddlePaddle/PaddleOCR-VL contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/PaddlePaddle/PaddleOCR-VL .
 You can inspect the repository content at https://hf.co/PaddlePaddle/PaddleOCR-VL.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y



Pipeline initialized and ready!


In [12]:
# ============================================
# STEP 5: Run OCR on the Pharmaceutical SDF
# ============================================

print("Running OCR on your pharmaceutical document...")

# Format the input message for the VLM
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": image_path},
            {"type": "text", "text": "OCR:"}
        ]
    }
]

# Run inference
paddle_result = pipe(messages)
paddle_text = paddle_result[0]['generated_text']

print("\nOCR complete!")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running OCR on your pharmaceutical document...


KeyboardInterrupt: 

In [ ]:
# ============================================
# STEP 6: Display Results
# ============================================
#
# Note: Because we are using a Vision-Language Model,
# the output is a continuous string of extracted text
# rather than polygon bounding boxes.
# ============================================

print("PaddleOCR (Hugging Face) Output:")
print("=" * 60)
print(paddle_text)

---

## Part 2: EasyOCR

EasyOCR is a simpler, faster alternative to PaddleOCR[cite: 2]. It's:
- Very easy to install and use[cite: 2]
- Good for scanned forms and simple layouts[cite: 2]
- Faster to set up than PaddleOCR[cite: 2]

Let's try it on the same document![cite: 2]

In [ ]:
# ============================================
# STEP 7: Install and Import EasyOCR
# ============================================

!pip install easyocr

import easyocr
from PIL import ImageDraw

print("\nEasyOCR installed and imported!")

In [ ]:
# ============================================
# STEP 8: Initialize EasyOCR Reader
# ============================================

print("Initializing EasyOCR reader...")

reader = easyocr.Reader(['en'])

print("EasyOCR reader ready!")

In [ ]:
# ============================================
# STEP 9: Run EasyOCR on the Pharmaceutical SDF
# ============================================

print("Running EasyOCR on your pharmaceutical document...")

easy_result = reader.readtext(image_path)

print(f"\nEasyOCR found {len(easy_result)} text regions!")

In [ ]:
# ============================================
# STEP 10: Display EasyOCR Results
# ============================================

print("Sample of EasyOCR Results:")
print("=" * 60)

for i, (bbox, text, confidence) in enumerate(easy_result[:15]):
    print(f"{i+1:2d}. [{confidence:.2%}] {text}")

if len(easy_result) > 15:
    print(f"\n... and {len(easy_result) - 15} more text regions!")

In [ ]:
# ============================================
# STEP 11: Visualize EasyOCR Results
# ============================================

img_easy = image.copy()
draw = ImageDraw.Draw(img_easy)

extracted_text = []

for (bbox, text, confidence) in easy_result:
    if confidence > 0.5:
        top_left = tuple(map(int, bbox[0]))
        bottom_right = tuple(map(int, bbox[2]))
        draw.rectangle([top_left, bottom_right], outline='red', width=2)
        extracted_text.append(text)

plt.figure(figsize=(15, 10))
plt.imshow(img_easy)
plt.title(f'EasyOCR Results - {len(extracted_text)} text segments found')
plt.axis('off')
plt.show()

---

## Part 3: Side-by-Side Comparison

Now let's compare PaddleOCR and EasyOCR results to see which works better for pharmaceutical documents![cite: 2]

In [ ]:
# ============================================
# STEP 12: Check for Key Pharmaceutical Terms
# ============================================
#
# WHAT WE'RE DOING:
# Checking if each engine found important pharmaceutical terms[cite: 2].
# ============================================

key_terms = ["CERTIFICATE", "LOT", "BATCH", "VENDOR", "EXPIRATION",
             "DATE", "COA", "MANUFACTURER", "STERIS", "PROCESSING", "QUALITY"]

# Get texts for matching
paddle_upper = paddle_text.upper()
easy_texts = [text.upper() for (_, text, _) in easy_result]

print("Key Pharmaceutical Terms Found:")
print("=" * 50)
print(f"{'Term':<15} {'Paddle (HF)':<15} {'EasyOCR':<12}")
print("-" * 50)

for term in key_terms:
    paddle_found = term in paddle_upper
    easy_found = any(term in text for text in easy_texts)

    paddle_mark = "Found" if paddle_found else "--"
    easy_mark = "Found" if easy_found else "--"

    print(f"{term:<15} {paddle_mark:<15} {easy_mark:<12}")